# Linear Regression: Zero to One
### Project: House price predictor on the California Housing dataset

This notebook is built step by step. **Step 1 (this section): Theory in plain English.**
No math background needed. Every symbol is defined the first time it appears.

## 1. Theory

### 1.1 What is regression?
**Regression** means predicting a *number* from other numbers.

- Predict a house's **price** (a number) from its **size, age, and location** (more numbers).
- The thing we predict is the **target** (also called the label), written $y$.
- The things we use to predict are the **features** (also called inputs), written $x$.
- **Training** means showing the computer many houses where we already know the price, so it learns the pattern.

If the target were a category ("cheap" / "expensive") it would be *classification*. A number means *regression*.

### 1.2 The line equation
You may remember from school: $y = mx + b$. Machine learning uses different letter names:

$$\hat{y} = w x + b$$

| Symbol | Name | Plain meaning |
|---|---|---|
| $\hat{y}$ ("y-hat") | prediction | what our model guesses |
| $y$ | true value | the real price |
| $x$ | feature | e.g. house size |
| $w$ | **weight** (slope) | how much the price changes when $x$ goes up by 1 |
| $b$ | **bias** (intercept) | the price when $x = 0$; the line's starting height |

Example: if $w = 50$ and $b = 20$ (in thousands of dollars), then a house of size 10 is predicted as $50 \cdot 10 + 20 = 520$.

With many features (size, age, rooms...) we just add more weights:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_d x_d + b$$

Here $d$ is the number of features. **Training means finding the best $w$'s and $b$.**

### 1.3 What does "best" mean? Error and residuals
For one house, the **residual** (error) is the gap between truth and guess:

$$\text{residual} = y - \hat{y}$$

A good line has small residuals across *all* houses. But we need ONE number that summarizes "how wrong is the line overall". That number is the **cost function**.

### 1.4 Why squared error?
The most common choice is to square each residual and average them. This is the **Mean Squared Error (MSE)**:

$$J(w, b) = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - \hat{y}_i\right)^2$$

Here $n$ is the number of houses, $i$ labels one house, and $\sum$ means "add up over all houses". $J$ is the **cost function** (also called loss function): a score where **lower = better**.

**Why square instead of just adding residuals?**
1. **Signs cancel.** A residual of +10 and one of -10 would sum to 0, which looks perfect but isn't. Squaring makes everything positive.
2. **Smooth.** The squared curve has no sharp corners, so calculus (and gradient descent) works nicely. Absolute value has a sharp corner at 0.
3. **Big errors are punished much more.** Being off by 2 costs $2^2 = 4$. Being off by 10 costs $10^2 = 100$. Five times the error costs **25 times** the penalty. The line is therefore pulled hard to avoid large misses. This is good if big misses are truly bad, and bad if your data has a few weird outliers, because they dominate the fit.

The code cell below shows this visually.

In [ ]:
# --- Setup: import the libraries we need for the theory demos ---
import numpy as np                 # NumPy: fast math on arrays of numbers
import matplotlib.pyplot as plt    # Matplotlib: plotting library

# Make plots a bit larger and easier to read
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
# --- Demo: how squared error punishes big mistakes more than absolute error ---

# A range of possible residuals (errors) from -10 to +10, 200 evenly spaced points
residuals = np.linspace(-10, 10, 200)

# Absolute error: just the size of the error, ignoring sign (grows in a straight line)
abs_error = np.abs(residuals)

# Squared error: error multiplied by itself (grows as a curve, faster and faster)
sq_error = residuals ** 2

# Plot both penalties against the size of the error
plt.plot(residuals, abs_error, label="Absolute error |e|")
plt.plot(residuals, sq_error, label="Squared error e^2")
plt.xlabel("Residual (y - y_hat)")
plt.ylabel("Penalty")
plt.title("Squared error punishes big mistakes far more")
plt.legend()
plt.show()

# Numeric version: compare penalties for a small and a big error
for e in [2, 10]:
    print(f"error = {e:>2}  ->  absolute penalty = {abs(e):>3},  squared penalty = {e**2:>3}")

**What the plot tells you:** near 0 the squared curve is *flatter* than the absolute line (small errors barely matter), but beyond about 1 it shoots upward. An error of 10 is only 5x worse than an error of 2 in absolute terms, but **25x worse** in squared terms.

### 1.5 The cost function as a landscape
Imagine only one unknown weight $w$ (ignore $b$). For each possible $w$ we can compute the cost $J(w)$. Plotting $J$ against $w$ gives a **bowl-shaped curve** (a parabola). The bottom of the bowl is the best $w$. With two unknowns the bowl becomes a 3D valley, and so on in higher dimensions. For linear regression with squared error the bowl always has **exactly one bottom** (the technical word is *convex*), so there are no tricky local traps.

The demo below builds a tiny dataset and draws this bowl.

In [ ]:
# --- Demo: the cost "bowl" for a tiny dataset with one feature and no bias ---

# Tiny dataset: the true relationship is y = 3 * x exactly
x = np.array([1.0, 2.0, 3.0, 4.0])   # features
y = 3.0 * x                           # targets

# Try many candidate weights from -2 to 8
w_values = np.linspace(-2, 8, 200)

# For each candidate w: predict (w * x), compute residuals, square them, average them
costs = [np.mean((y - w * x) ** 2) for w in w_values]

# Find the w with the smallest cost
best_w = w_values[np.argmin(costs)]

plt.plot(w_values, costs)
plt.axvline(best_w, color="red", linestyle="--", label=f"best w ~ {best_w:.2f}")
plt.xlabel("weight w")
plt.ylabel("cost J(w) (MSE)")
plt.title("Cost function is a bowl: one lowest point")
plt.legend()
plt.show()

### 1.6 Gradient descent: walking down the bowl
How does a computer find the bottom? **Gradient descent** is like standing on a foggy hillside and wanting to reach the valley:

1. Feel which direction the ground slopes **downhill** right where you stand.
2. Take a small step that way.
3. Repeat until the ground is flat (you're at the bottom).

The mathematical names:

- **Gradient** $\nabla J$: the slope of the cost. It points *uphill*, so we step in the *opposite* direction. (It is computed using derivatives, which measure "how fast does $J$ change if I nudge $w$ a tiny bit".)
- **Learning rate** $\eta$ ("eta"): the step size. Too small = painfully slow. Too big = you leap over the valley and may bounce out of control.
- **Iteration** (or epoch, when it passes over all data once): one repeat of the loop.

The update rule, applied to all weights at once:

$$w \leftarrow w - \eta \, \nabla J(w)$$

The arrow $\leftarrow$ means "replace $w$ with the value on the right".

### 1.7 The normal equation: skipping the walk
Because the bowl has a single bottom, we can *calculate* where it is directly: the bottom is where the slope is zero. Solving that gives a one-line formula, the **normal equation**. Below is the full derivation.

#### Setup in matrix form
Stack all houses into a **matrix** $X$ (a grid of numbers) with $n$ rows (houses) and $d+1$ columns (features). The extra column is all 1s so the bias $b$ can be treated as just another weight. Put all weights in a **vector** $w$ (a list of numbers, now including $b$). Then all predictions at once are:

$$\hat{y} = Xw$$

Notation: $X^T$ is the **transpose** (rows and columns swapped). $X^{-1}$ is the **inverse** (the matrix that "undoes" $X$, like dividing). $\|v\|^2 = v^T v$ means "sum of squares of the entries of $v$".

#### Step 1: write the cost in matrix form
$$J(w) = \frac{1}{n}(y - Xw)^T (y - Xw)$$
*Why:* $v^T v$ squares each entry and adds them up, which is exactly the sum of squared residuals.

#### Step 2: expand the brackets
$$J(w) = \frac{1}{n}\left(y^Ty - y^TXw - w^TX^Ty + w^TX^TXw\right)$$
*Why:* same as $(a-b)(a-b)$ but with matrices; we use $(Xw)^T = w^TX^T$.

#### Step 3: combine the two middle terms
$y^TXw$ is a single number, and a single number equals its own transpose, so $y^TXw = w^TX^Ty$:
$$J(w) = \frac{1}{n}\left(y^Ty - 2w^TX^Ty + w^TX^TXw\right)$$

#### Step 4: take the gradient (the slope with respect to $w$)
Two standard rules: the gradient of $w^Ta$ is $a$, and the gradient of $w^TAw$ is $2Aw$ (for symmetric $A$, and $X^TX$ is symmetric). The first term $y^Ty$ does not contain $w$, so its slope is 0:
$$\nabla J(w) = \frac{1}{n}\left(-2X^Ty + 2X^TXw\right)$$

#### Step 5: set the slope to zero (the bottom of the bowl)
$$-2X^Ty + 2X^TXw = 0 \;\Rightarrow\; X^TXw = X^Ty$$
*Why:* move one term across and divide both sides by 2. This is the **normal equation**.

#### Step 6: solve for $w$
Multiply both sides on the left by $(X^TX)^{-1}$:
$$\boxed{\,w = (X^TX)^{-1}X^Ty\,}$$

**Caveats you'll meet later:**
- It needs $X^TX$ to be invertible. If two features are copies/combinations of each other (**multicollinearity**) it breaks or becomes unstable.
- Inverting a big matrix is slow when there are very many features, which is when gradient descent wins.
- The same slope formula gives the gradient descent update: $\nabla J(w) = \frac{2}{n}X^T(Xw - y)$.

### 1.8 Step 1 summary
| Concept | One-line takeaway |
|---|---|
| Regression | Predict a number from features |
| Line | $\hat{y} = w_1x_1 + \dots + b$; training finds $w, b$ |
| Squared error | Positive, smooth, punishes big misses (error x5 means penalty x25) |
| Cost function | One score for the whole line, lower is better, bowl-shaped |
| Gradient descent | Repeated small downhill steps, controlled by the learning rate |
| Normal equation | Jump straight to the bottom: $w = (X^TX)^{-1}X^Ty$ |

**Next up (Step 2):** exploring the California Housing data with plots.

## 2. Exploratory Data Analysis (EDA)

**EDA** means *looking at your data before modeling*. We check what each column looks like, how columns relate, and what is weird. Models only learn what the data shows, so surprises found here save hours later.

### 2.1 Load the data
The **California Housing** dataset has 20,640 *block groups* (a block group is a small neighborhood of roughly 1,000 people from the 1990 census). One row = one block group. Columns:

| Column | Meaning |
|---|---|
| `MedInc` | median income in the block group (units: $10,000) |
| `HouseAge` | median age of houses (years) |
| `AveRooms` | average rooms per household |
| `AveBedrms` | average bedrooms per household |
| `Population` | people living in the block group |
| `AveOccup` | average people per household |
| `Latitude`, `Longitude` | location |
| **`MedHouseVal`** | **target**: median house value (units: $100,000) |

In [ ]:
# --- Imports for EDA ---
import numpy as np                       # numerical arrays
import pandas as pd                      # tables (DataFrames)
import matplotlib.pyplot as plt          # plotting
import seaborn as sns                    # nicer statistical plots built on matplotlib
from sklearn.datasets import fetch_california_housing   # dataset loader

sns.set_theme(style="whitegrid")         # clean plot style with light grid lines

In [ ]:
# --- Load the dataset as a pandas DataFrame ---
# as_frame=True gives us a DataFrame instead of raw NumPy arrays
# (the first run downloads the data; it is then cached on your machine)
housing = fetch_california_housing(as_frame=True)

# housing.frame contains the 8 feature columns plus the target column 'MedHouseVal'
df = housing.frame

# Show the first 5 rows to get a feel for the data
print("Shape (rows, columns):", df.shape)
df.head()

In [ ]:
# --- Basic health check ---
# .info() lists each column, its data type, and how many values are non-missing
df.info()

# Count missing values per column (we expect all zeros for this dataset)
print("\nMissing values per column:\n", df.isna().sum())

# .describe() gives count, mean, std (spread), min, quartiles (25/50/75%), max
# Look at 'max' vs '75%' to spot columns with extreme values
df.describe().T

**What this tells you:** there are no missing values (good). But look at `max` vs `75%`: `AveRooms` max is about 140 and `AveOccup` max is over 1,000, while their typical values are around 5 and 3. Those are almost certainly odd blocks (e.g. vacation areas or group housing). We'll see them clearly in the plots below.

### 2.2 Distributions (histograms)
A **histogram** splits a column's range into bins and counts how many rows fall in each. It shows the *shape* of the data: symmetric, skewed (a long tail on one side), or having spikes.

In [ ]:
# --- Histograms of every column ---
# df.hist draws one histogram per column; bins=40 means 40 bars each
axes = df.hist(bins=40, figsize=(14, 9), color="steelblue", edgecolor="white")

plt.suptitle("Distribution of every column", y=1.02, fontsize=14)  # overall title
plt.tight_layout()                                                  # avoid overlapping labels
plt.show()

**What each histogram tells you:**
- `MedInc`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`: **right-skewed** (most values small, a long tail of large ones). Linear regression prefers roughly symmetric data, so later we may apply a log transform.
- `HouseAge`: fairly flat with a spike at 52, meaning ages above 52 were lumped into 52 (**capping**).
- `MedHouseVal` (target): a tall spike at the far right (5.0 = $500,000). The data was **capped** there, so true values above $500k were recorded as 5.0. A line can't learn the real prices of those houses.
- `Latitude`/`Longitude`: two clusters, roughly the Bay Area and Los Angeles.

In [ ]:
# --- Quantify the target cap ---
# Count how many block groups have the maximum price value
cap_value = df["MedHouseVal"].max()                       # the largest target value
n_capped = (df["MedHouseVal"] == cap_value).sum()         # how many rows equal it
print(f"Max value = {cap_value}  ->  {n_capped} rows ({n_capped/len(df):.1%}) sit exactly at the cap")

### 2.3 Correlations
**Correlation** (Pearson's $r$) is a number from -1 to +1 describing how strongly two columns move together along a straight line:
- $+1$: when one goes up, the other always goes up (perfect straight line).
- $0$: no linear relationship.
- $-1$: when one goes up, the other goes down.

A **heatmap** colors a table of all pairwise correlations. **Caution:** correlation shows association, not cause, and it only detects *straight-line* relationships.

In [ ]:
# --- Correlation heatmap ---
corr = df.corr()                                    # table of pairwise correlations

plt.figure(figsize=(9, 7))
# annot=True prints the numbers; cmap 'coolwarm' = blue (negative) to red (positive)
# vmin/vmax fix the color scale to the full -1..1 range so colors are comparable
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation between columns")
plt.show()

# Rank features by how correlated they are with the target
print(corr["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False))

**What the heatmap tells you:**
- `MedInc` has the strongest link to price (about +0.69): richer neighborhoods have pricier homes. It will be our most important feature.
- `AveRooms` and `AveBedrms` are strongly correlated with *each other*. Two features carrying similar information is **multicollinearity**, which we'll test with VIF in Step 5.
- `Latitude` and `Longitude` are strongly (negatively) correlated with each other because California runs diagonally.

In [ ]:
# --- Scatter plots: feature vs target ---
# Use a random sample of 3000 points; plotting all 20,640 would just be a blob
sample = df.sample(3000, random_state=42)    # random_state makes the sample reproducible

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["MedInc", "HouseAge", "AveRooms"]):
    ax.scatter(sample[col], sample["MedHouseVal"], alpha=0.3, s=10)   # alpha = transparency
    ax.set_xlabel(col)
    ax.set_ylabel("MedHouseVal")
    ax.set_title(f"{col} vs price")
plt.tight_layout()
plt.show()

**What the scatter plots tell you:**
- `MedInc` vs price: a clear upward trend (linear regression suits this), a horizontal row of dots at 5.0 (the cap), and the spread gets wider as income grows. That fanning-out is the heteroscedasticity we test in Step 5.
- `HouseAge` vs price: almost a cloud, so a weak linear relationship.
- `AveRooms` vs price: a few extreme x values squash everything else into a thin band, which shows the outlier problem.

### 2.4 Outliers
An **outlier** is a value far from the rest. Outliers matter for regression because squared error lets them pull the line towards them. A **box plot** shows the middle 50% of data as a box; dots beyond the "whiskers" are flagged outliers. The **IQR rule** (IQR = Q3 - Q1, the box height) flags points below $Q1 - 1.5\cdot IQR$ or above $Q3 + 1.5\cdot IQR$.

In [ ]:
# --- Box plots of the skewed columns (separate axes since scales differ wildly) ---
cols = ["AveRooms", "AveBedrms", "Population", "AveOccup"]
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, col in zip(axes, cols):
    sns.boxplot(y=df[col], ax=ax, color="lightcoral")   # one box per column
    ax.set_title(col)
plt.tight_layout()
plt.show()

# --- Count outliers per column using the IQR rule ---
def iqr_outlier_count(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)       # 25th and 75th percentiles
    iqr = q3 - q1                                      # the box height
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr      # whisker limits
    return ((s < lower) | (s > upper)).sum()           # count values outside the limits

print("IQR-rule outlier counts:")
for col in df.columns:
    print(f"  {col:12s} {iqr_outlier_count(df[col]):5d}")

**What the box plots tell you:** the boxes are squashed flat near the bottom because a handful of extreme values (e.g. an `AveRooms` of 140) stretch the axis. Those points are real-world oddities, not typical houses.

**What to do (we'll use this later):** options are (1) remove clearly implausible rows, (2) **clip** (cap) values at a percentile, (3) log-transform skewed columns, (4) leave them and use a robust model. We'll decide when we model.

### 2.5 Geography
Latitude and longitude plotted as x/y make a rough map of California. Coloring by price shows location effects that a straight-line model on the raw numbers can't fully capture.

In [ ]:
# --- Map-like scatter: longitude (x) vs latitude (y), colored by price ---
plt.figure(figsize=(8, 7))
sc = plt.scatter(df["Longitude"], df["Latitude"],
                 c=df["MedHouseVal"],        # color each point by price
                 cmap="viridis", s=4, alpha=0.5)
plt.colorbar(sc, label="MedHouseVal ($100k)")  # color legend
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("House prices across California")
plt.show()

**What the map tells you:** the coastline is visible, and the expensive (bright) areas hug the coast near the Bay Area and Los Angeles, while inland is cheaper. Price depends on location in a *non-linear* way, so a plain linear model will have some limits here. That's a useful thing to know before we model.

### 2.6 Step 2 summary
| Finding | Consequence for modeling |
|---|---|
| No missing values | No imputation needed |
| Skewed features + extreme outliers | Consider log transform / clipping |
| Target capped at 5.0 | Model can't learn prices above $500k; expect errors there |
| `MedInc` is the top predictor | Expect a large coefficient |
| `AveRooms` / `AveBedrms` correlated | Check multicollinearity (VIF) later |
| Variance grows with income | Check heteroscedasticity later |

**Next up (Step 3):** building linear regression from scratch with NumPy.

## 3. Linear Regression From Scratch (NumPy only)

Now we turn the theory into code. We implement the **normal equation** and **gradient descent** ourselves using only NumPy, then check they agree with scikit-learn.

### 3.1 Prepare the data
Two preparation ideas, defined:

- **Train/test split:** we hide a portion of the data (the **test set**) during training and use it afterwards to measure how well the model handles *unseen* houses. Scoring on training data would be like grading a student on the exact questions they memorized. We use 80% to train, 20% to test.
- **Standardization (scaling):** subtract each column's mean and divide by its standard deviation (**std**, the typical spread), so every feature has mean 0 and std 1. This makes gradient descent converge faster and makes coefficients comparable. **Important:** compute the mean/std from the *training* data only and reuse them on the test data. Using test data to compute them would be **data leakage** (the test set secretly influences training).

In [ ]:
# --- Split and standardize (using NumPy for the maths) ---
from sklearn.model_selection import train_test_split   # only used for the random split

# Separate features (everything except the target) from the target column
X = df.drop(columns="MedHouseVal").values     # shape (20640, 8): NumPy array of features
y = df["MedHouseVal"].values                  # shape (20640,):  NumPy array of prices
feature_names = list(df.drop(columns="MedHouseVal").columns)   # remember column names

# 80% train / 20% test; random_state=42 makes the split identical every run
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Mean and std computed from TRAINING data only (axis=0 -> one value per column)
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)

# Standardize: subtract mean, divide by std. Test uses the TRAIN mu and sigma.
X_train_s = (X_train - mu) / sigma
X_test_s = (X_test - mu) / sigma

print("Train shape:", X_train_s.shape, " Test shape:", X_test_s.shape)
print("Train feature means (should be ~0):", np.round(X_train_s.mean(axis=0), 3))
print("Train feature stds  (should be ~1):", np.round(X_train_s.std(axis=0), 3))

### 3.2 (a) Normal equation
From Step 1: $w = (X^TX)^{-1}X^Ty$, where $X$ has an extra column of 1s for the bias.

Implementation note: instead of literally computing the inverse (slow and numerically fragile), we rearrange to $X^TXw = X^Ty$ and ask NumPy to **solve** that system with `np.linalg.solve`, which is the better practice. We also show the literal inverse version to prove they agree.

In [ ]:
# --- Normal equation from scratch ---
def add_bias_column(X):
    # Put a column of ones on the left so weight[0] acts as the bias b
    ones = np.ones((X.shape[0], 1))            # n rows, 1 column of 1s
    return np.hstack([ones, X])                # glue columns side by side

def normal_equation(X, y):
    Xb = add_bias_column(X)                    # shape (n, d+1)
    A = Xb.T @ Xb                              # X^T X  (@ is matrix multiplication)
    b = Xb.T @ y                               # X^T y
    return np.linalg.solve(A, b)               # solve A w = b for w

# Fit on the standardized training data
w_normal = normal_equation(X_train_s, y_train)

# Same answer via the literal formula (X^T X)^-1 X^T y, for comparison
Xb_train = add_bias_column(X_train_s)
w_inverse = np.linalg.inv(Xb_train.T @ Xb_train) @ Xb_train.T @ y_train

print("Max difference solve vs inverse:", np.abs(w_normal - w_inverse).max())

# Display the learned weights next to feature names
print("\nbias (intercept):", round(w_normal[0], 4))
for name, w in zip(feature_names, w_normal[1:]):
    print(f"{name:12s} {w:8.4f}")

**Reading the weights:** features are standardized, so each weight means "price change (in $100k) when this feature rises by one standard deviation, holding the others fixed". The bias equals the average training price (because the features are centered). Notice `Latitude` and `Longitude` have large negative weights and `MedInc` a large positive one.

### 3.3 (b) Gradient descent
From Step 1: the slope of the cost is $\nabla J(w) = \frac{2}{n}X^T(Xw - y)$, and we repeat $w \leftarrow w - \eta\nabla J(w)$.

Recipe per iteration: (1) predict with the current $w$, (2) compute the gradient, (3) step downhill, (4) record the cost so we can plot the learning curve.

In [ ]:
# --- Gradient descent from scratch ---
def gradient_descent(X, y, lr=0.1, n_iters=2000):
    Xb = add_bias_column(X)                    # add bias column of ones
    n, d = Xb.shape                            # n houses, d weights (incl. bias)
    w = np.zeros(d)                            # start with all weights = 0
    losses = []                                # store the cost at each iteration

    for _ in range(n_iters):
        preds = Xb @ w                         # current predictions
        errors = preds - y                     # (prediction - truth) for each house
        losses.append(np.mean(errors ** 2))    # MSE cost J(w) at this step
        grad = (2 / n) * (Xb.T @ errors)       # gradient of MSE w.r.t. every weight
        w = w - lr * grad                      # step opposite to the gradient

    return w, losses

# Run it: learning rate 0.1, 2000 iterations
w_gd, losses = gradient_descent(X_train_s, y_train, lr=0.1, n_iters=2000)

# Plot the loss against iteration number (the "learning curve")
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title("Gradient descent: loss vs iteration")
plt.yscale("log")                              # log scale so the late, tiny improvements are visible
plt.show()

print("Final training MSE:", round(losses[-1], 5))
print("Max |w_gd - w_normal|:", np.abs(w_gd - w_normal).max())

**What the plot tells you:** the loss falls steeply at first (big slope = big steps) then flattens as we approach the bottom of the bowl (slope near 0 = tiny steps). Flat means converged.

### 3.4 Effect of the learning rate
Same algorithm, three step sizes. Watch how the choice of $\eta$ changes the story.

In [ ]:
# --- Compare learning rates ---
for lr in [0.001, 0.05, 0.3]:
    _, l = gradient_descent(X_train_s, y_train, lr=lr, n_iters=300)   # fewer iterations for a quick look
    plt.plot(l, label=f"lr = {lr}")

plt.xlabel("Iteration")
plt.ylabel("Training MSE")
plt.title("Learning rate matters")
plt.yscale("log")
plt.legend()
plt.show()

**What it tells you:** the tiny rate (0.001) is still crawling after 300 steps, while the larger rates reach the bottom fast. If the rate were too big (try `lr=1.5`), the steps overshoot and the loss *increases* or explodes. Try it yourself.

### 3.5 Verify against scikit-learn
`LinearRegression` is scikit-learn's ready-made solver. If our code is right, all three should produce the same weights and the same test predictions.

In [ ]:
# --- Verify our implementations against sklearn ---
from sklearn.linear_model import LinearRegression

# Fit sklearn on the same standardized data (it handles the bias itself via intercept_)
sk = LinearRegression().fit(X_train_s, y_train)
w_sklearn = np.concatenate([[sk.intercept_], sk.coef_])    # same layout as ours: [bias, weights...]

# Compare the weights
print("Max |normal eq - sklearn|     :", np.abs(w_normal - w_sklearn).max())
print("Max |grad descent - sklearn|  :", np.abs(w_gd - w_sklearn).max())

# Compare predictions on the unseen test set
Xb_test = add_bias_column(X_test_s)
pred_normal = Xb_test @ w_normal
pred_gd = Xb_test @ w_gd
pred_sk = sk.predict(X_test_s)

print("Max prediction gap (normal vs sklearn):", np.abs(pred_normal - pred_sk).max())
print("Max prediction gap (GD vs sklearn)    :", np.abs(pred_gd - pred_sk).max())

# np.allclose is True when all numbers match within a tiny tolerance
print("Normal equation matches sklearn:", np.allclose(w_normal, w_sklearn))
print("Gradient descent matches sklearn:", np.allclose(w_gd, w_sklearn, atol=1e-3))

# A first look at test performance: R^2 (1.0 = perfect, 0 = no better than predicting the mean)
ss_res = np.sum((y_test - pred_normal) ** 2)         # residual sum of squares
ss_tot = np.sum((y_test - y_test.mean()) ** 2)       # total variation around the mean
print("Test R^2 (from scratch):", round(1 - ss_res / ss_tot, 4))

**What it tells you:** the gaps are essentially zero, so the from-scratch code reproduces sklearn exactly. Test $R^2$ is about 0.58. The model explains a bit over half the variation in prices, consistent with the limits we spotted in EDA (capping, location non-linearity). We'll improve our understanding of these limits later.

### 3.6 Step 3 summary
| Method | Idea | Pros | Cons |
|---|---|---|---|
| Normal equation | Solve slope = 0 directly | Exact, no tuning | Slow with huge feature counts; fails if $X^TX$ is singular |
| Gradient descent | Walk downhill iteratively | Scales to big data; basis of deep learning | Needs a learning rate and enough iterations |

**Next up (Step 4):** sklearn `LinearRegression`, Ridge and Lasso, with cross-validation.

## 4. The scikit-learn Way: LinearRegression, Ridge and Lasso

In real projects we use tested libraries instead of hand-written solvers. **scikit-learn** (sklearn) gives a uniform pattern: create a model object, call `.fit(X, y)` to train, `.predict(X)` to predict.

### 4.1 Plain LinearRegression
This is the ordinary least squares (OLS) fit from Step 3, now in one line. We'll also compute our first test-set scores. (Metrics get a full treatment in Step 7.)

In [ ]:
# --- Imports for sklearn modeling ---
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# Helper: print test RMSE and R^2 for any fitted model
def report(name, model, Xte, yte):
    pred = model.predict(Xte)                              # predictions on unseen data
    rmse = np.sqrt(mean_squared_error(yte, pred))          # RMSE: typical error in $100k units
    r2 = r2_score(yte, pred)                               # R^2: share of variation explained
    print(f"{name:22s} test RMSE = {rmse:.4f}   test R^2 = {r2:.4f}")

# Fit plain OLS on the standardized training data
ols = LinearRegression().fit(X_train_s, y_train)
report("LinearRegression (OLS)", ols, X_test_s, y_test)

### 4.2 Regularization: why and how
**Overfitting** means a model memorizes quirks (noise) in the training data and then does worse on new data. Linear models overfit when there are many features, few rows, or highly correlated features: the weights can become huge and unstable to chase noise.

**Regularization** adds a *penalty for large weights* to the cost function, so the model must justify every bit of weight it uses:

| Model | Cost function | Penalty on weights | Effect |
|---|---|---|---|
| OLS | $\text{MSE}$ | none | can overfit |
| **Ridge** (L2) | $\text{MSE} + \alpha\sum w_j^2$ | squared size | shrinks all weights smoothly toward 0, rarely exactly 0 |
| **Lasso** (L1) | $\text{MSE} + \alpha\sum \lvert w_j\rvert$ | absolute size | shrinks and sets some weights to **exactly 0** (automatic feature selection) |

- $\alpha$ ("alpha") is the **hyperparameter** (a setting *you* choose, not learned from data) controlling the penalty strength. $\alpha = 0$ gives OLS. A huge $\alpha$ forces all weights to 0 (underfit: too simple).
- The bias $b$ is not penalized.
- This is why we standardized: the penalty treats all weights equally, so features must be on the same scale.
- The trade-off is **bias vs variance**: regularization adds a little bias (systematic error) to reduce variance (sensitivity to the particular training sample).

### 4.3 Choosing alpha with cross-validation
**Cross-validation (CV)** estimates how well a setting generalizes without touching the test set. With 5-fold CV: split the training data into 5 parts (**folds**); train on 4, validate on the 1 left out; rotate so each fold is the validation set once; average the 5 scores. We try many alphas and pick the one with the best average validation score.

In [ ]:
# --- Ridge: cross-validation curve over many alphas ---
alphas = np.logspace(-3, 3, 30)        # 30 values from 0.001 to 1000, evenly spaced on a log scale

cv_mse = []                             # average validation MSE for each alpha
for a in alphas:
    # cross_val_score returns NEGATIVE MSE (sklearn convention: higher score = better), so flip the sign
    scores = cross_val_score(Ridge(alpha=a), X_train_s, y_train, cv=5, scoring="neg_mean_squared_error")
    cv_mse.append(-scores.mean())

plt.semilogx(alphas, cv_mse, marker="o")                      # log x-axis because alpha spans orders of magnitude
plt.axvline(alphas[np.argmin(cv_mse)], color="red", linestyle="--", label="best alpha")
plt.xlabel("alpha (log scale)")
plt.ylabel("5-fold CV MSE")
plt.title("Ridge: choosing alpha by cross-validation")
plt.legend()
plt.show()

# RidgeCV does the same search automatically and refits on all training data with the winner
ridge = RidgeCV(alphas=alphas, cv=5).fit(X_train_s, y_train)
print("Best Ridge alpha:", round(ridge.alpha_, 4))
report("Ridge (CV alpha)", ridge, X_test_s, y_test)

In [ ]:
# --- Lasso: LassoCV searches its own alpha path with 5-fold CV ---
lasso = LassoCV(alphas=np.logspace(-5, 0, 40), cv=5, max_iter=20000).fit(X_train_s, y_train)
print("Best Lasso alpha:", round(lasso.alpha_, 6))
report("Lasso (CV alpha)", lasso, X_test_s, y_test)

# Plot the CV error curve: mse_path_ has shape (n_alphas, n_folds)
plt.semilogx(lasso.alphas_, lasso.mse_path_.mean(axis=1), marker="o")   # mean over the 5 folds
plt.axvline(lasso.alpha_, color="red", linestyle="--", label="best alpha")
plt.xlabel("alpha (log scale)")
plt.ylabel("5-fold CV MSE")
plt.title("Lasso: choosing alpha by cross-validation")
plt.legend()
plt.show()

**What these plots tell you:** the CV error is lowest at a *small* alpha, then rises as alpha grows (too much shrinkage = underfitting). The optimum is small because we have 16,000 training rows and only 8 features, which is a setting where plain OLS is already stable.

### 4.4 Coefficient shrinkage plots
A **coefficient path** plot shows each weight as alpha increases. This is the clearest picture of what regularization does.

In [ ]:
# --- Coefficient paths: fit Ridge and Lasso over a range of alphas and record the weights ---
ridge_alphas = np.logspace(-2, 4, 40)
lasso_alphas = np.logspace(-4, 0.5, 40)

ridge_coefs = [Ridge(alpha=a).fit(X_train_s, y_train).coef_ for a in ridge_alphas]
lasso_coefs = [Lasso(alpha=a, max_iter=20000).fit(X_train_s, y_train).coef_ for a in lasso_alphas]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, alph, coefs, title in [(axes[0], ridge_alphas, np.array(ridge_coefs), "Ridge (L2): smooth shrinkage"),
                               (axes[1], lasso_alphas, np.array(lasso_coefs), "Lasso (L1): weights hit exactly zero")]:
    for j, name in enumerate(feature_names):
        ax.semilogx(alph, coefs[:, j], label=name)       # one line per feature
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("alpha (log scale)")
    ax.set_ylabel("coefficient value")
    ax.set_title(title)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

**What the plots tell you:**
- **Ridge:** all lines glide smoothly toward 0 as alpha grows, but none reach it exactly. Correlated features (like `Latitude`/`Longitude`) can shrink at different speeds or even swap direction.
- **Lasso:** lines drop to *exactly* zero one by one. The features that survive the longest (here `MedInc`) are the most useful. This is automatic feature selection.

In [ ]:
# --- Side-by-side coefficient table: OLS vs Ridge vs Lasso (CV-tuned) ---
coef_table = pd.DataFrame({
    "OLS": ols.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_,
}, index=feature_names)
print(coef_table.round(4))

### 4.5 When does regularization actually help? A demo
On this dataset the three models score almost identically, which is an honest and important lesson: **regularization is not magic**. It helps when overfitting is a real risk. Let's create that situation on purpose: use only 150 training rows and add 60 columns of pure random noise (features with no relationship to price).

In [ ]:
# --- Demo: many junk features + few rows = overfitting danger ---
rng = np.random.default_rng(0)                         # reproducible random generator

n_small = 150                                           # tiny training set
noise_train = rng.normal(size=(n_small, 60))            # 60 junk features for training rows
noise_test = rng.normal(size=(X_test_s.shape[0], 60))   # 60 junk features for test rows

# Real 8 features + 60 junk features
Xtr_big = np.hstack([X_train_s[:n_small], noise_train])
ytr_small = y_train[:n_small]
Xte_big = np.hstack([X_test_s, noise_test])

ols_big = LinearRegression().fit(Xtr_big, ytr_small)
ridge_big = RidgeCV(alphas=np.logspace(-1, 4, 40), cv=5).fit(Xtr_big, ytr_small)
lasso_big = LassoCV(alphas=np.logspace(-4, 0, 40), cv=5, max_iter=50000).fit(Xtr_big, ytr_small)

report("OLS  (68 features)", ols_big, Xte_big, y_test)
report("Ridge (68 features)", ridge_big, Xte_big, y_test)
report("Lasso (68 features)", lasso_big, Xte_big, y_test)

# How many of the 60 junk features did Lasso zero out?
junk_zero = np.sum(lasso_big.coef_[8:] == 0)
print(f"\nLasso set {junk_zero} of the 60 junk features exactly to 0")

**What it tells you:** plain OLS does clearly worse here because it uses the junk columns to memorize noise in 150 rows. Ridge and Lasso recover much of the lost accuracy, and Lasso additionally discards most junk features. That is regularization doing its job.

### 4.6 Step 4 summary
| Concept | Takeaway |
|---|---|
| sklearn pattern | `fit` then `predict`; the same results as our scratch code |
| Regularization | Penalize big weights to reduce overfitting |
| Ridge vs Lasso | Ridge shrinks smoothly; Lasso also zeros features out |
| alpha | Hyperparameter chosen by cross-validation, never by the test set |
| Reality check | With plenty of data and few features, OLS is already fine |

**Next up (Step 5):** assumption checks (residuals, Q-Q, heteroscedasticity, VIF).

## 5. Assumption Checks

Linear regression gives *an* answer for any data, but the answer is only trustworthy if some conditions hold. These are the **assumptions**, remembered as **LINE** (plus one more):

| Letter | Assumption | Plain meaning | If violated |
|---|---|---|---|
| **L** | Linearity | The true relationship is a straight line in the features | Predictions are systematically off in some regions |
| **I** | Independence | Rows' errors don't influence each other (e.g. no time-order patterns) | Confidence intervals become too optimistic |
| **N** | Normality | Residuals look bell-shaped | Matters mostly for p-values/intervals, less for predictions |
| **E** | Equal variance (**homoscedasticity**) | Error size is the same everywhere | Some predictions are far less reliable than others; intervals are wrong |
| + | No multicollinearity | Features aren't near-copies of each other | Coefficients become unstable and uninterpretable |

**Residuals** (from Step 1) are our best window: if the assumptions hold, residuals look like random, shapeless noise.

### 5.1 Fit the baseline model and collect residuals

In [ ]:
# --- Imports for diagnostics ---
import statsmodels.api as sm                                   # statistics-focused regression library
from scipy import stats                                         # statistical tools (Q-Q plot)
from statsmodels.stats.diagnostic import het_breuschpagan       # heteroscedasticity test
from statsmodels.stats.outliers_influence import variance_inflation_factor   # VIF
from statsmodels.stats.stattools import durbin_watson           # independence (autocorrelation) check

# Fitted values = the model's predictions on the training data; residual = truth - prediction
fitted = ols.predict(X_train_s)
resid = y_train - fitted

# Helper that draws the three standard residual diagnostics for any fitted/residual pair
def residual_diagnostics(fitted, resid, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # (1) Residuals vs fitted: should be a flat, even band around 0
    axes[0].scatter(fitted, resid, alpha=0.2, s=8)
    axes[0].axhline(0, color="red")                             # the zero-error line
    axes[0].set_xlabel("Fitted value"); axes[0].set_ylabel("Residual")
    axes[0].set_title("Residuals vs fitted")

    # (2) Histogram of residuals: should look roughly like a symmetric bell
    axes[1].hist(resid, bins=60, color="steelblue", edgecolor="white")
    axes[1].set_xlabel("Residual"); axes[1].set_title("Residual distribution")

    # (3) Q-Q plot: compares residual quantiles to a perfect normal distribution
    stats.probplot(resid, dist="norm", plot=axes[2])            # points on the line = normal
    axes[2].set_title("Q-Q plot")

    fig.suptitle(title, y=1.03)
    plt.tight_layout()
    plt.show()

residual_diagnostics(fitted, resid, "Baseline OLS diagnostics")

### 5.2 Reading the three plots
- **Residuals vs fitted** (checks Linearity and Equal variance): a healthy plot is a horizontal, even-width band centered on 0. Here you will see (a) a *fan* shape where residual spread grows with the fitted value (heteroscedasticity) and (b) a **diagonal stripe** of points on the left/bottom, which are the houses whose price was capped at 5.0: the model predicts something lower, the truth is stuck at the cap, so the residual falls on a straight line.
- **Histogram:** roughly bell-shaped but with a longer right tail.
- **Q-Q plot** (**Q-Q** = quantile-quantile): each dot compares a residual to where a perfectly normal residual of that rank would sit. Dots on the red line = normal. Here the *ends curve away*, meaning heavier tails than normal: more extreme errors than a bell curve predicts.

### 5.3 Formal test for unequal variance
Plots are subjective, so we add a number. The **Breusch-Pagan test** checks whether residual size depends on the features. Its **p-value** is the probability of seeing a pattern this strong if variance were truly equal. Convention: **p < 0.05 means "reject equal variance"** (heteroscedasticity detected).

In [ ]:
# --- Breusch-Pagan test for heteroscedasticity ---
# statsmodels needs the feature matrix WITH a constant (bias) column
exog = sm.add_constant(X_train_s)

# Returns: LM statistic, LM p-value, F statistic, F p-value
lm_stat, lm_p, f_stat, f_p = het_breuschpagan(resid, exog)
print(f"Breusch-Pagan LM statistic = {lm_stat:.1f},  p-value = {lm_p:.2e}")
print("Heteroscedasticity detected!" if lm_p < 0.05 else "No evidence of heteroscedasticity.")

# Durbin-Watson: ~2 means no autocorrelation; near 0 or 4 means a problem.
# (Our rows aren't time-ordered, so this is shown only for completeness.)
print("Durbin-Watson:", round(durbin_watson(resid), 3))

**What it tells you:** a tiny p-value, so heteroscedasticity is confirmed statistically, not just visually. (With 16,000 rows, tests detect even small effects, so always judge the *plot* alongside the p-value.)

### 5.4 A violation and its fix: transform the target
**Why a fan appears:** errors in house prices are *proportional* (a $50k miss on a $100k home vs on a $400k home). Proportional errors grow with the level, so variance grows with the fitted value.

**Fix:** model the **logarithm** of price. On a log scale, "x% error" becomes a constant additive error, which turns multiplicative noise into equal-variance noise. Predictions are converted back with `exp`.

In [ ]:
# --- Fix: fit on log(price) ---
y_train_log = np.log(y_train)                        # natural log of the target (all prices are > 0)

ols_log = LinearRegression().fit(X_train_s, y_train_log)
fitted_log = ols_log.predict(X_train_s)
resid_log = y_train_log - fitted_log

residual_diagnostics(fitted_log, resid_log, "After log-transforming the target")

# Re-run Breusch-Pagan on the new residuals
_, lm_p_log, _, _ = het_breuschpagan(resid_log, exog)
print(f"Breusch-Pagan p-value after log fix: {lm_p_log:.2e}")

# Compare performance on the ORIGINAL price scale (convert predictions back with exp)
pred_test_log = np.exp(ols_log.predict(X_test_s))
print("Test R^2 original target       :", round(r2_score(y_test, ols.predict(X_test_s)), 4))
print("Test R^2 log target (raw exp)  :", round(r2_score(y_test, pred_test_log), 4))
print("Largest back-transformed prediction ($100k):", round(pred_test_log.max(), 1))

# Clip predictions into the range of prices seen in training (0 .. 5.0)
pred_test_log_clipped = np.clip(pred_test_log, 0, y_train.max())
print("Test R^2 log target (clipped)  :", round(r2_score(y_test, pred_test_log_clipped), 4))

**What it tells you (read carefully, this one has a twist):**
- The residual shape improves: the Breusch-Pagan statistic still flags unequal variance (with 16,000 rows it detects even small effects), so the log transform *reduces* the problem but does not eliminate it. The price cap (the diagonal stripe) is a data problem no transform removes.
- **The raw back-transformed R² is terrible (negative!).** Why? `exp()` *amplifies* errors. A few rows have extreme feature values (e.g. `AveBedrms` about 76 standard deviations from average). A straight line extrapolates wildly there, and `exp` turns a big log-prediction into an absurd price (several million $100k units). One crazy prediction ruins R² because squared error punishes big misses so hard (Step 1!).
- **Remedy:** clip predictions to the plausible range (0 to the max training price). After clipping, R² is *better* than plain OLS.
- **Lessons:** fixing one assumption can introduce a new risk; always re-evaluate on the original scale; and outliers in the *features* (Step 2) matter.

### 5.5 Multicollinearity and VIF
**Multicollinearity:** two or more features carry nearly the same information. The model can't decide how to split credit between them, so coefficients become wildly unstable (they can even flip sign) even though predictions stay fine.

**VIF (Variance Inflation Factor):** for each feature, predict it from all the *other* features and see how well that works. $\text{VIF}_j = \frac{1}{1 - R_j^2}$. Rule of thumb: **1 = independent, 5+ = concerning, 10+ = serious**.

In [ ]:
# --- VIF for each feature of the real data ---
Xc = sm.add_constant(X_train_s)                       # VIF needs the constant column included

vif = pd.DataFrame({
    "feature": feature_names,
    # index j+1 because column 0 is the constant we added
    "VIF": [variance_inflation_factor(Xc, j + 1) for j in range(len(feature_names))],
}).sort_values("VIF", ascending=False)
print(vif.round(2).to_string(index=False))

**What it tells you:** `Latitude` and `Longitude` (about 9) and `AveRooms` / `AveBedrms` (about 7) sit in the "concerning" zone, because California's geography makes lat/lon move together and bigger houses have more of both rooms and bedrooms. None pass 10, so the coefficients are usable but should be interpreted with care (e.g. the odd positive `AveBedrms` weight next to a negative `AveRooms` weight is a collinearity symptom).

### 5.6 What a *severe* violation looks like
Let's create one deliberately: add a column that is a near-copy of `AveRooms` (the original times 2 plus a tiny bit of noise). Then fit the same model on five different small random subsamples (300 rows each) of the training data and watch the coefficients. (With very large samples the instability hides; it shows up when data is limited.)

In [ ]:
# --- Create a near-duplicate feature ---
rng = np.random.default_rng(1)
rooms_idx = feature_names.index("AveRooms")                       # position of AveRooms
dup = 2 * X_train_s[:, rooms_idx] + rng.normal(scale=0.01, size=len(X_train_s))   # almost an exact copy
X_dup = np.hstack([X_train_s, dup.reshape(-1, 1)])                # 9 features now

# VIF of the two guilty columns
Xdc = sm.add_constant(X_dup)
print("VIF AveRooms     :", round(variance_inflation_factor(Xdc, rooms_idx + 1), 1))
print("VIF duplicate col:", round(variance_inflation_factor(Xdc, X_dup.shape[1]), 1))

# Fit on five random 300-row subsamples and compare the two coefficients each time
for k in range(5):
    idx = rng.choice(len(X_dup), size=300, replace=False)        # pick 300 random rows
    m = LinearRegression().fit(X_dup[idx], y_train[idx])
    r = Ridge(alpha=10).fit(X_dup[idx], y_train[idx])
    print(f"subsample {k+1}: OLS coefs (AveRooms, dup) = {m.coef_[rooms_idx]:7.2f}, {m.coef_[-1]:7.2f}"
          f"   |  Ridge = {r.coef_[rooms_idx]:6.2f}, {r.coef_[-1]:6.2f}")

**What it tells you:** the VIFs explode to around 40,000. The two OLS coefficients have opposite signs and are far larger than the true combined effect, and they swing wildly from one subsample to the next (even changing sign). The model is "cancelling" the two columns against each other, so interpretation is meaningless even though predictions may stay fine. Ridge's coefficients stay small and consistent.

**Fixes:**
1. **Drop** one of the redundant features (simplest, best when features are truly duplicates).
2. **Combine** them (e.g. average them, or use PCA).
3. **Regularize** with Ridge (shown above: the Ridge coefficients stay small and stable across halves because the penalty refuses huge opposing weights).
4. Collect more diverse data.

### 5.7 Step 5 summary
| Check | Tool | Our finding | Remedy |
|---|---|---|---|
| Linearity / equal variance | Residuals vs fitted | Fan shape + capped stripe | Log-transform target; transform skewed features |
| Normality | Histogram, Q-Q | Heavy tails | Transform; robust methods; large n helps |
| Heteroscedasticity | Breusch-Pagan | Detected (still flagged after log fix, but milder) | Log target (clip predictions!), weighted least squares, robust standard errors |
| Multicollinearity | VIF | Moderate for lat/lon; severe in our demo | Drop, combine, Ridge |

**Next up (Step 6):** failure modes: feature leakage and data drift.

## 6. Failure-Mode Demos

A model can score beautifully in your notebook and still fail in the real world. Two of the most common reasons are **feature leakage** and **data drift**. We will cause each one on purpose so you recognize it later.

### 6.1 (a) Feature leakage
**Feature leakage** means the model sees information during training that *won't be available* (or won't mean the same thing) when it makes real predictions. Often the leaky feature is a disguised copy of the answer, or something recorded *after* the outcome happened.

**Our leaky column:** imagine someone adds `AssessedValue`, the value the county tax office assigned *after the house sold*, which is within a few percent of the sale price. It looks like a harmless column in the spreadsheet, but at prediction time (before the sale) it does not exist yet.

In [ ]:
# --- Build a leaky feature: a noisy copy of the target ---
rng = np.random.default_rng(7)                              # reproducible randomness

# AssessedValue = true price times a random factor near 1 (3% noise): almost the answer itself
leak_all = y * rng.normal(loc=1.0, scale=0.03, size=len(y))

# Split the leak column with the SAME random_state=42 so rows line up with X_train / X_test
leak_train, leak_test = train_test_split(leak_all, test_size=0.2, random_state=42)

# Append the leaky column as a 9th feature (raw, unscaled values)
X_train_leak = np.hstack([X_train, leak_train.reshape(-1, 1)])
X_test_leak = np.hstack([X_test, leak_test.reshape(-1, 1)])

# Standardize using TRAIN statistics only (same discipline as before)
from sklearn.preprocessing import StandardScaler            # sklearn's built-in scaler (same maths as Step 3)
scaler_leak = StandardScaler().fit(X_train_leak)
Xtr_l = scaler_leak.transform(X_train_leak)
Xte_l = scaler_leak.transform(X_test_leak)

# Fit and score
leaky_model = LinearRegression().fit(Xtr_l, y_train)
report("Model WITH leaky column", leaky_model, Xte_l, y_test)
report("Honest model (no leak)", ols, X_test_s, y_test)

# Which feature does the model rely on? Look at the standardized coefficients
for name, w in zip(feature_names + ["AssessedValue (LEAK)"], leaky_model.coef_):
    print(f"{name:22s} {w:8.3f}")

**What it tells you:** test $R^2$ jumps from about 0.58 to about **0.996**. A nearly perfect score on a hard problem is a *red flag*, not a celebration. Look at the coefficients: every real feature collapsed to about 0 and `AssessedValue` carries all the weight. The model isn't predicting prices, it is reading the answer off a column.

Now simulate **production**: a new house arrives *before it has sold*, so `AssessedValue` doesn't exist. The pipeline still expects 9 inputs, so someone fills the missing value with a placeholder (the training average).

In [ ]:
# --- Production simulation: the leaky column is unavailable at prediction time ---
X_test_prod = X_test_leak.copy()                             # copy so we don't modify the original

# Fill the missing AssessedValue with the training mean (a typical "default" a pipeline might use)
X_test_prod[:, -1] = leak_train.mean()

report("Leaky model in PRODUCTION", leaky_model, scaler_leak.transform(X_test_prod), y_test)

**What it tells you:** $R^2$ falls to roughly **0** (no better than always guessing the average price), far *below* the honest 0.58 model. The leaky model learned to ignore the real signals, so when the leak disappears it has nothing left.

**Why it fails in production:** validation metrics are only meaningful if the test set mimics real deployment. The leak made the notebook score look great but described a world that doesn't exist.

**How to catch leakage:**
1. Be suspicious of scores that are too good to be true.
2. Check for a single feature dominating the coefficients, or a feature correlating almost perfectly with the target.
3. For every feature ask: *"Will I know this value at the exact moment I make a prediction?"*
4. Beware **preprocessing leakage** too: computing scaling/imputation statistics on the full dataset *before* splitting lets test information seep into training. (That's why Step 3 computed `mu` and `sigma` from the training rows only.)

### 6.2 (b) Data drift
**Data drift** means the real-world data after deployment differs from the data the model was trained on. Two flavors:
- **Covariate shift:** the *inputs* change (e.g. newer, wealthier neighborhoods appear).
- **Concept drift:** the *relationship* between inputs and price changes (e.g. the same income now buys a very different house price in a different region).

A linear model learns one fixed set of weights, so it can't adapt. We will train on one slice of California and test on a shifted slice, comparing with a fair in-distribution test.

In [ ]:
# --- Drift demo: train on one slice, test on a different slice ---
def drift_experiment(name, train_mask, test_mask):
    # train_mask / test_mask are True/False arrays choosing which rows belong to each slice
    Xa, ya = X[train_mask], y[train_mask]                    # the slice we train on
    # Hold out 20% of the TRAINING slice for a fair "no drift" score
    Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(Xa, ya, test_size=0.2, random_state=0)

    sc = StandardScaler().fit(Xa_tr)                         # scale using training-slice statistics only
    m = LinearRegression().fit(sc.transform(Xa_tr), ya_tr)

    # Score 1: unseen rows from the SAME slice (what a normal notebook would report)
    p_in = m.predict(sc.transform(Xa_val))
    # Score 2: rows from the SHIFTED slice (what production would look like)
    Xb, yb = X[test_mask], y[test_mask]
    p_out = m.predict(sc.transform(Xb))

    return {
        "scenario": name,
        "R2 same-slice": r2_score(ya_val, p_in),
        "R2 shifted": r2_score(yb, p_out),
        "MAE same-slice": np.mean(np.abs(ya_val - p_in)),
        "MAE shifted": np.mean(np.abs(yb - p_out)),
        "mean error (bias) shifted": np.mean(yb - p_out),    # actual - predicted: NEGATIVE = model OVER-predicts on average
    }

# Masks built from the full DataFrame columns
age = df["HouseAge"].values
lon = df["Longitude"].values
lat = df["Latitude"].values

results = pd.DataFrame([
    # Train on NEWER neighborhoods (age <= 20), test on OLD ones (age >= 35)
    drift_experiment("new -> old neighborhoods", age <= 20, age >= 35),
    # Train on the south/east (longitude > -119, mostly Southern California), test on the northern coast (Bay Area)
    drift_experiment("south/east -> Bay Area", lon > -119, (lon < -122) & (lat > 37)),
]).set_index("scenario")
print(results.round(3).T)

**What it tells you:** in both scenarios the score on the *same slice* looks decent ($R^2 \approx 0.55$-$0.6$) but drops on the shifted slice ($R^2$ = -0.20 for new-to-old, which is *worse than guessing the average*, and 0.21 for south/east-to-Bay-Area). The error (MAE, the average miss in $100k) grows by 40% to 100%. For the Bay Area scenario the **bias** (actual minus predicted) is large and negative: the model systematically *over-predicts* by about $83k, because weights learned in the south/east (where location and income relate to price in one way) don't transfer to the Bay Area. Note the new-to-old scenario has almost no bias but much noisier errors: drift can hurt in different ways.

Let's visualize the shifted failure.

In [ ]:
# --- Visualize the coast drift: predicted vs actual, same-slice vs shifted ---
train_mask, test_mask = lon > -119, (lon < -122) & (lat > 37)
Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(X[train_mask], y[train_mask], test_size=0.2, random_state=0)
sc = StandardScaler().fit(Xa_tr)
m = LinearRegression().fit(sc.transform(Xa_tr), ya_tr)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel 1: predicted vs actual on the same slice (points near the red diagonal = good)
axes[0].scatter(ya_val, m.predict(sc.transform(Xa_val)), alpha=0.2, s=8)
axes[0].plot([0, 5], [0, 5], "r--")
axes[0].set_title("Same slice (south/east)")

# Panel 2: predicted vs actual on the shifted slice (points fall ABOVE the line = over-prediction)
axes[1].scatter(y[test_mask], m.predict(sc.transform(X[test_mask])), alpha=0.2, s=8, color="darkorange")
axes[1].plot([0, 5], [0, 5], "r--")
axes[1].set_title("Shifted slice (Bay Area)")

# Panel 3: the INPUT distribution shift (covariate shift): median income in each slice
axes[2].hist(df.loc[train_mask, "MedInc"], bins=40, alpha=0.6, density=True, label="train slice")
axes[2].hist(df.loc[test_mask, "MedInc"], bins=40, alpha=0.6, density=True, label="shifted slice")
axes[2].set_title("MedInc distribution: train vs shifted")
axes[2].legend()

for ax in axes[:2]:
    ax.set_xlabel("Actual price"); ax.set_ylabel("Predicted price")
plt.tight_layout()
plt.show()

**What the plots tell you:** left, predictions hug the diagonal (reasonable). Middle, the shifted points sit *above* the diagonal: the model confidently overestimates (x-axis = truth, y-axis = prediction). Right, the input distributions differ, which is covariate shift on top of the changed relationship (concept drift).

**Mitigations:**
1. **Monitor** in production: track input statistics and the error over time (you only know the truth later, so also track input drift).
2. **Retrain** regularly on recent data.
3. **Train on representative data** covering the regions/conditions you'll serve (or add region features).
4. Use **time/group-aware validation** (split by region or date) instead of a random split, so your notebook score reflects the shift you'll really face.

Quick check of the third mitigation: add a small sample of the new region to the training data.

In [ ]:
# --- Mitigation: include a bit of data from the shifted region in training ---
shift_idx = np.where(test_mask)[0]                           # row numbers of the Bay Area slice
rng2 = np.random.default_rng(0)
rng2.shuffle(shift_idx)                                      # shuffle them randomly
seen, unseen = shift_idx[:500], shift_idx[500:]              # 500 Bay Area rows join training; the rest are for testing

X_mix = np.vstack([Xa_tr, X[seen]])                          # original training rows + 500 Bay Area rows
y_mix = np.concatenate([ya_tr, y[seen]])

sc2 = StandardScaler().fit(X_mix)
m2 = LinearRegression().fit(sc2.transform(X_mix), y_mix)

before = r2_score(y[unseen], m.predict(sc.transform(X[unseen])))
after = r2_score(y[unseen], m2.predict(sc2.transform(X[unseen])))
print(f"Bay Area R^2 before: {before:.3f}   after adding 500 Bay Area rows: {after:.3f}")

**What it tells you:** a small amount of representative data improves coastal performance substantially. Linear regression isn't broken by drift, it just only knows what it was shown.

### 6.3 Step 6 summary
| Failure | Symptom in notebook | Real-world result | Defence |
|---|---|---|---|
| Feature leakage | Absurdly high score (R² 0.996), one dominant coefficient | Collapses once the leaky feature is unavailable (R² about 0) | Ask "available at prediction time?"; split before preprocessing; distrust perfect scores |
| Data drift | Fine on same-distribution test | R² drops, errors become biased | Monitor, retrain, representative data, group/time-aware splits |

**Next up (Step 7):** metrics (MAE, RMSE, R²) and when each one misleads.

## 7. Metrics: MAE, RMSE and R²

A **metric** is a number that scores how good predictions are. Different metrics answer different questions, and each can mislead. Let $e_i = y_i - \hat{y}_i$ be the residual of house $i$ and $n$ the number of test houses.

| Metric | Formula | Units | Plain meaning |
|---|---|---|---|
| **MAE** (Mean Absolute Error) | $\frac{1}{n}\sum \lvert e_i\rvert$ | same as target ($100k) | "On average, I'm off by this much" |
| **MSE** (Mean Squared Error) | $\frac{1}{n}\sum e_i^2$ | target units squared | The cost function we trained on; hard to read |
| **RMSE** (Root Mean Squared Error) | $\sqrt{\text{MSE}}$ | same as target | Like MAE but big misses count much more |
| **R²** (coefficient of determination) | $1 - \frac{\sum e_i^2}{\sum (y_i - \bar{y})^2}$ | none (unitless) | Share of the variation in price the model explains; 1 = perfect, 0 = no better than always predicting the average $\bar{y}$, negative = worse than that |

($\bar{y}$ is the average true price.) In R², the top is the model's total squared error and the bottom is the squared error of the dumbest baseline (guess the average). So R² asks: "how much better than guessing the average am I?"

### 7.1 Compute them by hand and with sklearn

In [ ]:
# --- Metrics from scratch vs sklearn, on the OLS test predictions ---
from sklearn.metrics import mean_absolute_error

pred = ols.predict(X_test_s)               # predictions on the unseen test set
e = y_test - pred                          # residuals (truth - prediction)

mae_s = np.mean(np.abs(e))                                  # average absolute miss
mse_s = np.mean(e ** 2)                                     # average squared miss
rmse_s = np.sqrt(mse_s)                                     # square root puts it back in $100k units
r2_s = 1 - np.sum(e ** 2) / np.sum((y_test - y_test.mean()) ** 2)   # 1 - (model error / baseline error)

print(f"From scratch : MAE={mae_s:.4f}  RMSE={rmse_s:.4f}  R2={r2_s:.4f}")
print(f"sklearn      : MAE={mean_absolute_error(y_test, pred):.4f}  "
      f"RMSE={np.sqrt(mean_squared_error(y_test, pred)):.4f}  R2={r2_score(y_test, pred):.4f}")

# Translate into dollars: the target is in units of $100,000
print(f"\nIn dollars: typical miss (MAE) ~ ${mae_s * 100_000:,.0f}, RMSE ~ ${rmse_s * 100_000:,.0f}")

**What it tells you:** both sets agree exactly. The average miss is about $53,000 and RMSE is a bit larger (about $75,000). **RMSE is always $\ge$ MAE**; the bigger the gap, the more your errors are dominated by a few large misses.

### 7.2 When MAE and RMSE mislead
- **MAE** treats every dollar of error equally. It can hide a model that is *occasionally catastrophic*.
- **RMSE** squares errors, so a few huge misses dominate it. It can make a model look bad because of a handful of outliers, or reflect real costs if big misses are truly expensive.

Demo: two models with the **identical MAE** but very different behavior.

In [ ]:
# --- Same MAE, different RMSE ---
# Model A: 100 houses, every prediction off by exactly 1
errors_a = np.ones(100)
# Model B: 96 houses predicted perfectly, 4 houses off by 25 (a few disasters)
errors_b = np.concatenate([np.zeros(96), np.full(4, 25.0)])

for name, err in [("Model A (steady)", errors_a), ("Model B (rare disasters)", errors_b)]:
    print(f"{name:26s} MAE = {np.mean(np.abs(err)):5.2f}   RMSE = {np.sqrt(np.mean(err**2)):5.2f}")

# Real-data version: inject 5 absurd predictions into our honest test predictions
pred_bad = pred.copy()
pred_bad[:5] += 50                                           # five houses predicted $5 million too high
print(f"\nHonest model          : MAE = {mean_absolute_error(y_test, pred):.3f}  "
      f"RMSE = {np.sqrt(mean_squared_error(y_test, pred)):.3f}")
print(f"With 5 crazy outliers : MAE = {mean_absolute_error(y_test, pred_bad):.3f}  "
      f"RMSE = {np.sqrt(mean_squared_error(y_test, pred_bad)):.3f}")

**What it tells you:** A and B tie on MAE (1.0) but RMSE exposes B (5.0 vs 1.0). Conversely, on real data just 5 absurd predictions out of 4,128 raise MAE by only about 11% (0.53 to 0.59) but more than double RMSE (0.75 to 1.89), the same effect we saw with the log-target model in Step 5.

**Rule of thumb:** report **both**. Use MAE when you want a robust, easy-to-explain "typical miss". Use RMSE when large errors are disproportionately costly (e.g. pricing a house hugely wrong loses a client).

### 7.3 When R² misleads
**(a) A high R² doesn't mean the model is right.** A straight line can score a high R² on clearly curved data:

In [ ]:
# --- R^2 can be high even when the model shape is wrong ---
rng = np.random.default_rng(3)
x_c = np.linspace(0, 10, 200)                                # 200 evenly spaced inputs
y_c = x_c ** 2 + rng.normal(scale=3, size=200)               # true relationship is a CURVE (quadratic) plus noise

line = LinearRegression().fit(x_c.reshape(-1, 1), y_c)       # but we fit a straight line
fit_c = line.predict(x_c.reshape(-1, 1))
print("R^2 of a straight line on curved data:", round(r2_score(y_c, fit_c), 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x_c, y_c, s=10); axes[0].plot(x_c, fit_c, "r")        # data + the line
axes[0].set_title("High R^2 but systematically wrong shape")
axes[1].scatter(fit_c, y_c - fit_c, s=10); axes[1].axhline(0, color="red")  # residual plot reveals the curve
axes[1].set_title("Residuals show a U-shaped pattern")
axes[1].set_xlabel("Fitted"); axes[1].set_ylabel("Residual")
plt.tight_layout(); plt.show()

**What it tells you:** R² is around 0.9, which sounds great, yet the residual plot shows a clear U-shape: the line is too high in the middle and too low at the ends. R² never tells you the model is *appropriate*, so always pair it with residual plots (Step 5).

**(b) In-sample R² rewards complexity.** On *training* data, R² never goes down when you add features, even junk ones. Revisit the 68-feature, 150-row overfitting demo from Step 4:

In [ ]:
# --- Training R^2 vs test R^2 for the overfit model from Step 4 ---
train_r2 = r2_score(ytr_small, ols_big.predict(Xtr_big))      # score on the data the model memorized
test_r2 = r2_score(y_test, ols_big.predict(Xte_big))          # score on unseen data
print(f"OLS with 68 features, 150 rows -> train R^2 = {train_r2:.3f}, test R^2 = {test_r2:.3f}")

**What it tells you:** the training R² is high (0.84) while the test R² is near zero (0.04). **Always report R² on held-out data**, never on training data alone. (An *adjusted R²* exists that penalizes extra features, but a proper test set is the better defence.)

**(c) R² depends on how spread out the data is, not just on model quality.** The same model, evaluated on different subsets, can have a *lower* error but a *lower* R² too, because R² compares to a baseline whose difficulty depends on the spread of prices:

In [ ]:
# --- Same model, different subsets: accuracy (RMSE) and R^2 can disagree ---
medinc_test = X_test[:, feature_names.index("MedInc")]        # raw (unscaled) median income of each test house
low = medinc_test < np.median(medinc_test)                    # the poorer half of neighborhoods
high = ~low                                                    # the richer half

for name, mask in [("All test rows", np.ones_like(low, dtype=bool)), ("Lower-income half", low), ("Higher-income half", high)]:
    rmse = np.sqrt(mean_squared_error(y_test[mask], pred[mask]))
    print(f"{name:20s} RMSE = {rmse:.3f}   R^2 = {r2_score(y_test[mask], pred[mask]):.3f}   price std = {y_test[mask].std():.3f}")

**What it tells you:** on the lower-income half the model is actually *more accurate* in real units (RMSE 0.66 vs 0.82), yet its R² is *worse* (0.34 vs 0.47). That subset has a smaller spread of prices (see the `price std` column), so there is less variation to "explain" and the baseline of guessing the average is already hard to beat. Comparing R² across datasets with different spreads is unfair, whereas RMSE/MAE stay in real units.

### 7.4 One more trap: the average hides *where* you fail
A single number averages over everything. Slicing errors by price band shows where the model struggles:

In [ ]:
# --- MAE by true-price quartile ---
bands = pd.qcut(y_test, 4, labels=["cheapest 25%", "25-50%", "50-75%", "most expensive 25%"])   # split into 4 equal groups by true price
abs_err = np.abs(y_test - pred)
summary = pd.DataFrame({"band": bands, "abs_err": abs_err, "signed_err": y_test - pred}).groupby("band", observed=True).agg(
    MAE=("abs_err", "mean"),                  # typical miss in this band
    bias=("signed_err", "mean"),              # truth - prediction: positive = model too low on average
)
print(summary.round(3))

**What it tells you:** the model **over-predicts cheap houses and under-predicts expensive ones** (bias flips sign across bands) and the expensive band has the largest errors. This is the regression-to-the-mean effect, plus the $500k cap from EDA. The overall MAE of about 0.53 hides this.

### 7.5 Which metric when?
| Situation | Prefer | Watch out for |
|---|---|---|
| Explaining the "typical miss" to a non-technical person | **MAE** | Hides rare huge errors |
| Big errors are very costly / you want to punish outliers | **RMSE** | A few outliers dominate it |
| Comparing to a naive baseline on the *same* test set | **R²** | High R² on wrong shape; depends on data spread; must be held-out |
| Targets spanning orders of magnitude | Relative errors (e.g. MAPE) or log-scale metrics | Absolute metrics overweight the expensive items |
| Always | Several metrics + residual plots + error slices | Single-number summaries |

**Next up (Step 8):** 8 "prove it" questions, 3 coding exercises, and an extension project.

## 8. Prove It: Questions, Exercises and an Extension Project

Time to test yourself. **No answers are included in this notebook.** When you have attempted a question, ask your mentor for the answer.

### 8.1 Eight "Prove it" questions
"Prove it" means: don't just state the answer. Back it with a short derivation, a numerical experiment (use the notebook's variables like `X_train_s`, `y_train`), or both. Write your answer in a new cell under each question.

**Q1. Squared vs absolute error.** (a) Show with numbers that making an error 5x larger multiplies the squared penalty by 25 but the absolute penalty by only 5. (b) For the list `[1, 2, 3, 4, 100]`, find by brute force (try many candidate constants $c$) which $c$ minimizes the mean *squared* error and which minimizes the mean *absolute* error. What do the two winners correspond to, and what does that say about outliers?

**Q2. The gradient.** For one feature and no bias, $J(w) = \frac{1}{n}\sum (y_i - w x_i)^2$. Derive $\frac{dJ}{dw}$ by hand, then prove numerically that your formula matches a finite-difference estimate $\frac{J(w+h) - J(w-h)}{2h}$ with a tiny $h$ (such as $10^{-6}$).

**Q3. Does scaling change OLS?** Fit `LinearRegression` on the raw features (`X_train`) and on the standardized ones (`X_train_s`). Prove whether the *predictions* on the test set are identical, and whether the *coefficients* are identical. Explain the difference in terms of what a coefficient means.

**Q4. Singular matrix.** Add an exact copy of one feature as an extra column and compute $X^TX$. Prove it is singular (use its rank or determinant), then show what `np.linalg.inv` and `np.linalg.solve` each do with it. What does `np.linalg.lstsq` do instead?

**Q5. R² can't go down (in-sample).** Starting from the 8 real features, add junk random columns one at a time and refit OLS. Plot training R² and test R² against the number of junk columns. Prove that training R² never decreases, and describe what the test R² does and why.

**Q6. Why Ridge needs scaling.** Multiply the `Population` column by 1000 (on raw data, before any scaling). Fit OLS and Ridge(alpha=1) on this raw version and on the original raw version. Prove which of the two models gives different *predictions* after the rescale, and explain why a penalty on weights is sensitive to units.

**Q7. Residuals sum to zero?** Prove numerically that OLS residuals on the training set sum to (almost exactly) zero *when an intercept is fitted*. Then fit with `fit_intercept=False` on the raw (unstandardized) data and show that the property fails. Explain the connection to the normal equation $X^T(y - Xw) = 0$ and the column of ones.

**Q8. RMSE vs MAE.** Prove (mathematically or by a convincing experiment on 1000 random error vectors of different shapes) that RMSE $\geq$ MAE always. State exactly when they are equal. Then explain what the ratio RMSE/MAE tells you about an error distribution.

### 8.2 Three coding exercises
Each exercise gives a starter function with a docstring. Replace the `return None` line with your implementation. The check cell under each one tells you whether it passes (it prints a "not yet implemented" message until you do). Use only NumPy inside your functions unless told otherwise. Helpers from earlier steps are available: `add_bias_column`, `X_train_s`, `X_test_s`, `y_train`, `y_test`.

#### Exercise 1: Stochastic (mini-batch) gradient descent
In Step 3 gradient descent used **all** rows each iteration (full-batch). **Mini-batch gradient descent** uses a small random subset (a *batch*) each step, which is cheaper per step and is how deep learning is trained. An **epoch** is one pass over all the data. Implement it with a learning rate that decays over time, then compare its loss curve with the full-batch one.

In [ ]:
# --- Exercise 1 starter: mini-batch gradient descent ---
def minibatch_gd(X, y, lr=0.05, epochs=30, batch_size=64, decay=0.0, seed=0):
    '''
    Fit linear regression weights with mini-batch gradient descent.

    X          : standardized feature matrix WITHOUT the bias column
    y          : targets
    lr         : starting learning rate
    epochs     : number of full passes over the data
    batch_size : rows per update
    decay      : learning rate at epoch t is lr / (1 + decay * t)
    Returns    : (w, epoch_losses) where w = [bias, w1..wd] and epoch_losses is the
                 full-data MSE recorded at the end of every epoch.

    Hints: shuffle the row order each epoch (np.random.default_rng(seed).permutation),
    slice it into batches, compute the gradient on each batch only, then update w.
    '''
    # TODO: implement
    return None

# --- Exercise 1 check ---
result = minibatch_gd(X_train_s, y_train)
if result is None:
    print("Exercise 1: not yet implemented")
else:
    w_mb, mb_losses = result
    gap = np.abs(w_mb - w_normal).max()
    print(f"Max |w_minibatch - w_normal| = {gap:.4f}; final MSE = {mb_losses[-1]:.4f} (exact optimum ~ {losses[-1]:.4f})")
    print("PASS" if gap < 0.1 else "Close? Try more epochs or a smaller lr; compare the loss curve to full-batch GD")

#### Exercise 2: Ridge from scratch
The Ridge cost is a squared-error term plus a penalty on the weights, and the bias $w_0$ is **not** penalized. Be careful with conventions: scikit-learn's `Ridge` minimizes

$$\sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha\sum_{j\ge1} w_j^2$$

(a *sum* of squared errors, not a mean). Set the gradient to zero, as we did for the normal equation in Step 1, to derive the closed-form solution, paying attention to the penalty matrix (an identity matrix whose first diagonal entry, for the bias, must be 0). Implement it and verify against `sklearn.linear_model.Ridge`.

In [ ]:
# --- Exercise 2 starter: closed-form Ridge ---
def ridge_normal_equation(X, y, alpha):
    '''
    Closed-form Ridge regression.

    X     : standardized features WITHOUT bias column
    y     : targets
    alpha : penalty strength (use the same meaning as sklearn's Ridge)
    Returns w = [bias, w1..wd], with the bias NOT penalized.
    '''
    # TODO: implement (hint: build the penalty matrix with np.eye and zero out its [0, 0] entry)
    return None

# --- Exercise 2 check against sklearn ---
w_r = ridge_normal_equation(X_train_s, y_train, alpha=10.0)
if w_r is None:
    print("Exercise 2: not yet implemented")
else:
    sk_r = Ridge(alpha=10.0).fit(X_train_s, y_train)
    target = np.concatenate([[sk_r.intercept_], sk_r.coef_])
    print("Max difference vs sklearn Ridge:", np.abs(w_r - target).max())
    print("PASS" if np.allclose(w_r, target, atol=1e-6) else "Not matching yet: re-check which convention sklearn uses for alpha")

#### Exercise 3: K-fold cross-validation from scratch (and a group-aware version)
Implement **k-fold CV** yourself, without sklearn's splitters, and use it with your Ridge from Exercise 2 to pick alpha. Then add a **grouped** variant where entire latitude bands are held out together (like the drift demo of Step 6), and compare which alpha each version prefers and how optimistic the random version is.

In [ ]:
# --- Exercise 3 starter: k-fold CV and grouped CV ---
def kfold_cv_rmse(X, y, alpha, k=5, groups=None, seed=0):
    '''
    Return the mean validation RMSE of your Ridge (from Exercise 2) across k folds.

    groups : None -> random folds (shuffle row indices, split into k parts).
             array of group labels (one per row) -> keep every group entirely inside ONE fold
             (e.g. np.unique(groups), shuffle the groups, and assign them to folds).
    Steps per fold: train on the other folds, predict on the held-out fold, compute RMSE.
    '''
    # TODO: implement
    return None

# --- Exercise 3 check ---
lat_train = X_train[:, feature_names.index("Latitude")]                    # raw latitude of each training row
lat_band = np.digitize(lat_train, np.quantile(lat_train, [0.2, 0.4, 0.6, 0.8]))   # 5 latitude bands labelled 0..4

test_alphas = [0.01, 1, 100, 1000]
random_scores = [kfold_cv_rmse(X_train_s, y_train, a) for a in test_alphas]
if random_scores[0] is None or ridge_normal_equation(X_train_s, y_train, 1.0) is None:
    print("Exercise 3: not yet implemented (it also needs Exercise 2)")
else:
    group_scores = [kfold_cv_rmse(X_train_s, y_train, a, groups=lat_band) for a in test_alphas]
    for a, r_, g_ in zip(test_alphas, random_scores, group_scores):
        print(f"alpha={a:>7}: random-fold RMSE = {r_:.4f}   latitude-grouped RMSE = {g_:.4f}")
    print("Question to answer: why is the grouped RMSE usually higher, and which one is closer to production?")

### 8.3 Extension project: "Neighborhood-aware price model"
**Goal:** push a *linear* model as far as you can on this dataset without using trees or neural networks, and prove the improvement honestly.

**Baseline to beat:** test RMSE = 0.7456, R² = 0.5758 (Step 3/4).

**Milestones**
1. **Feature engineering:** add location features (e.g. K-Means clusters of latitude/longitude, one-hot encoded; distance to San Francisco and Los Angeles), polynomial terms of `MedInc`, and interaction features (e.g. `MedInc` x location cluster). Log-transform the skewed columns and clip the outliers you found in Step 2.
2. **Pipeline:** wrap everything in a single `sklearn.pipeline.Pipeline` (scaling, features, model) so that nothing is fitted on test data (no preprocessing leakage).
3. **Honest validation:** compare random K-fold with `GroupKFold` using location clusters as groups. Report both and explain the gap.
4. **Regularize:** tune Ridge/Lasso alpha by cross-validation on your expanded feature set and report which engineered features Lasso keeps.
5. **Diagnostics:** redo the residual plot, Q-Q plot, Breusch-Pagan and VIF on your final model. Did you fix or worsen each assumption?
6. **Failure testing:** retest your model on the drift slices from Step 6. Does a location-aware model reduce the Bay Area bias?
7. **Report:** a one-page summary table (MAE, RMSE, R² for baseline vs final) with plots, and a short paragraph on what you would monitor in production.

**Stretch:** serve the final pipeline with FastAPI (`joblib.dump` the pipeline to a `*.pkl` file, which `.gitignore` already excludes) and write a small script that simulates drift monitoring.

### 8.4 Where you are now
You have gone from the definition of a line, through the maths of the normal equation and gradient descent, to checking assumptions, breaking models on purpose, and judging them honestly. When you want the answers to the 8 questions, or feedback on your exercises, just ask.